# OCR Bilans Fiscaux Algériens — V8 — H100 + Qwen3.6-VL-27B

> **Objectif** : extraire les bilans fiscaux PDF → JSON complet + Excel récapitulatif + fichiers FORFAIT `.xlsx` par client/année avec contrôle de cohérence (rouge si incohérence).

## Nouveautés V8
| # | Changement | Détail |
|---|---|---|
| 1 | Template PREREMPLISSAGE_BILAN.xlsx | Chemin /mnt/Risk/forfait/ |
| 2 | TCR DEBIT/CREDIT | Extraction séparée debit_n/credit_n/debit_n1/credit_n1 |
| 3 | Contrôle cohérence | Si valeur extraite ≠ formule → cellule ROUGE + commentaire |
| 4 | Conversion KDZD | dinars ÷ 1000 |

## Livrables
| # | Livrable | Format |
|---|---|---|
| 1 | Base de données bilans | `.json` par PDF |
| 2 | Vue analyste | `.xlsx` (2 lignes/bilan) |
| 3 | FORFAIT par client | `FORFAIT Cas 1_{client}_{année}.xlsx` |

## Règle d'or
**Ne jamais inventer de valeur** : toute donnée absente/incertaine → `null` + anomalie signalée.

## Cellule 1 — Dépendances

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 1 — DÉPENDANCES | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
%pip install -q -U "transformers>=4.57.0" accelerate pymupdf pillow openpyxl psutil pandas
print('✅ OK')

## Cellule 2 — Imports

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 2 — IMPORTS | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
import time, json, re, gc, difflib, csv, shutil
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import fitz, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.comments import Comment
from openpyxl.utils import get_column_letter
print('✅ Imports OK')

## Cellule 3 — Config + FORFAIT_COORDS

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 3 — CONFIG | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'

DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_NEW_TOKENS = 3000
IMAGE_MAX_SIZE = 2024
MIN_PIXELS     = 4 * 32 * 32
MAX_PIXELS     = 2000 * 32 * 32
PDF_ZOOM       = 3.0
BLANK_THRESHOLD= 0.95
GPU_BATCH_SIZE = 8
ANNEE_ATTENDUE = None
TOLERANCE_KDZD = 0.01  # tolérance en KDZD pour le contrôle de cohérence

INPUT_DIR  = Path('/mnt/Risk/bilans_in')
OUTPUT_DIR = Path('/mnt/Risk/bilans_out')
JSON_DIR   = OUTPUT_DIR / 'json_bilans'
LOG_PATH   = OUTPUT_DIR / 'pipeline_bilans.log'
EXCEL_PATH = OUTPUT_DIR / f'bilans_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'

TEMPLATE_FORFAIT   = Path('/mnt/Risk/forfait/PREREMPLISSAGE_BILAN.xlsx')
FORFAIT_OUTPUT_DIR = OUTPUT_DIR / 'forfaits'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
FORFAIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))

# ─── STYLES ROUGE POUR INCOHÉRENCE ─────────────────────────
RED_FONT   = Font(color='FFFF0000', bold=True, name='Arial', size=10)
RED_FILL   = PatternFill(start_color='FFFFC7CE', end_color='FFFFC7CE', fill_type='solid')
RED_BORDER = Border(
    left=Side(style='thick', color='FFFF0000'),
    right=Side(style='thick', color='FFFF0000'),
    top=Side(style='thick', color='FFFF0000'),
    bottom=Side(style='thick', color='FFFF0000')
)
NORMAL_FONT = Font(name='Arial', size=10)

# ─── FORFAIT_COORDS (1-based, openpyxl) ─────────────────────
FORFAIT_COORDS = {
    'actif': {
        'sheet': 'Saisie actif',
        'cols_n':  {'brut': 3, 'amort': 4, 'net': 5},
        'cols_n1': {'brut': 6, 'amort': 7, 'net': 8},
        'nature_bilan':   (2, 2),
        'date_arrete_n':  (3, 2),
        'date_arrete_n1': (3, 2),
        'certifie_cac':   (5, 9),
        'rows': {
            'ecarts_acquisition_goodwill': 7,
            'immobilisations_incorporelles': 8,
            'terrains': 10,
            'batiments': 11,
            'autres_immobilisations_corporelles': 12,
            'immobilisations_en_concession': 13,
            'immobilisations_en_cours': 14,
            'titres_mis_en_equivalence': 16,
            'autres_participations_creances': 17,
            'autres_titres_immobilises': 18,
            'prets_actifs_financiers_non_courants': 19,
            'impots_differes_actif': 20,
            'total_actif_non_courant': 21,
            'stocks_encours': 23,
            'clients': 25,
            'autres_debiteurs': 26,
            'impots_assimiles_actif': 27,
            'autres_creances_assimiles': 28,
            'placements_financiers_courants': 30,
            'tresorerie_actif': 31,
            'total_actif_courant': 32,
            'total_general_actif': 33,
        }
    },
    'passif': {
        'sheet': 'Saisie passif',
        'col_n':  3,
        'col_n1': 4,
        'rows': {
            'capital_emis': 5,
            'capital_non_appele': 6,
            'primes_reserves': 7,
            'ecart_reevaluation': 8,
            'ecart_equivalence': 9,
            'resultat_net_passif': 10,
            'report_a_nouveau': 11,
            'part_societe_consolidante': 12,
            'part_minoritaires': 13,
            'total_capitaux_propres': 14,
            'emprunts_dettes_financieres': 16,
            'impots_differes_provisionnes': 17,
            'autres_dettes_non_courantes': 18,
            'provisions_produits_avance': 19,
            'total_passifs_non_courants': 20,
            'fournisseurs_rattaches': 22,
            'impots_passif': 23,
            'autres_dettes': 24,
            'tresorerie_passif': 25,
            'total_passifs_courants': 26,
            'total_general_passif': 27,
        }
    },
    'tcr': {
        'sheet': 'Saisie TCR',
        'cols_n':  {'debit': 3, 'credit': 4},
        'cols_n1': {'debit': 5, 'credit': 6},
        'rows': {
            'ventes_marchandises': 3,
            'produits_fabriques': 4,
            'prestations_services': 5,
            'ventes_travaux': 6,
            'produits_annexes': 7,
            'rabais_remises_ristournes_accordes': 8,
            'chiffre_affaires_net': 9,
            'production_stockee_destockee': 10,
            'production_immobilisee': 11,
            'subvention_exploitation': 12,
            'production_exercice': 13,
            'achats_marchandises_vendues': 14,
            'matieres_premieres': 15,
            'autres_approvisionnements': 16,
            'variation_stocks': 17,
            'achats_etudes_prestations': 18,
            'autres_consommations': 19,
            'rabais_remises_obtenus_achats': 20,
            'sous_traitance_generale': 21,
            'locations': 22,
            'entretien_reparations': 23,
            'primes_assurances': 24,
            'personnel_exterieur': 25,
            'remuneration_intermediaires': 26,
            'publicite': 27,
            'deplacements_missions': 28,
            'autres_services': 29,
            'rabais_remises_obtenus_services': 30,
            'consommations_exercice': 31,
            'valeur_ajoutee_exploitation': 32,
            'charges_personnel': 33,
            'impots_taxes_assimiles': 34,
            'excedent_brut_exploitation': 35,
            'autres_produits_operationnels': 36,
            'autres_charges_operationnelles': 37,
            'dotations_amortissements': 38,
            'provisions': 39,
            'pertes_valeur': 40,
            'reprises_pertes_valeur_provisions': 41,
            'resultat_operationnel': 42,
            'produits_financiers': 43,
            'charges_financieres': 44,
            'resultat_financier': 45,
            'resultat_ordinaire': 46,
            'elements_extraordinaires_produits': 47,
            'elements_extraordinaires_charges': 48,
            'resultat_extraordinaire': 49,
            'impots_exigibles_resultats': 50,
            'impots_differes_resultats': 51,
            'resultat_net_exercice': 52,
        }
    },
}

# ─── TABLE DE SENS TCR (pour ventilation si extraction en _n/_n1) ───
TCR_CREDIT = {
    'ventes_marchandises', 'produits_fabriques', 'prestations_services',
    'ventes_travaux', 'produits_annexes', 'chiffre_affaires_net',
    'production_stockee_destockee', 'production_immobilisee',
    'subvention_exploitation', 'production_exercice',
    'rabais_remises_obtenus_achats', 'rabais_remises_obtenus_services',
    'valeur_ajoutee_exploitation', 'excedent_brut_exploitation',
    'autres_produits_operationnels', 'reprises_pertes_valeur_provisions',
    'produits_financiers', 'elements_extraordinaires_produits',
}
TCR_DEBIT = {
    'rabais_remises_ristournes_accordes', 'achats_marchandises_vendues',
    'matieres_premieres', 'autres_approvisionnements', 'variation_stocks',
    'achats_etudes_prestations', 'autres_consommations',
    'sous_traitance_generale', 'locations', 'entretien_reparations',
    'primes_assurances', 'personnel_exterieur', 'remuneration_intermediaires',
    'publicite', 'deplacements_missions', 'autres_services',
    'consommations_exercice', 'charges_personnel', 'impots_taxes_assimiles',
    'autres_charges_operationnelles', 'dotations_amortissements',
    'provisions', 'pertes_valeur', 'charges_financieres',
    'elements_extraordinaires_charges', 'impots_exigibles_resultats',
    'impots_differes_resultats',
}
TCR_VARIABLE = {
    'resultat_operationnel', 'resultat_financier', 'resultat_ordinaire',
    'resultat_extraordinaire', 'resultat_net_exercice',
}

print(f'Device : {DEVICE} | Dossiers : {len(pdfs)} | Template : {TEMPLATE_FORFAIT.name}')

## Cellule 4 — Chargement modèle

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 4 — CHARGEMENT MODÈLE | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True,
                                          min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'left'

try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, device_map='auto',
    trust_remote_code=True, low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=True),
)
model.eval()
print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')

## Cellule 5 — Utilitaires PDF & inférence

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 5 — UTILITAIRES | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side: return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)

def estimate_skew(img):
    small = img.convert('L').copy(); small.thumbnail((500, 500))
    def score(a):
        r = np.array(small.rotate(a, expand=True, fillcolor=255)) < 128
        proj = r.sum(axis=1)
        return float((proj ** 2).sum())
    best = max(range(-12, 13, 2), key=score)
    best = max([best-1, best-0.5, best, best+0.5, best+1], key=score)
    return best if abs(best) >= 1 else 0.0

def deskew(img):
    a = estimate_skew(img)
    if a: img = img.rotate(a, expand=True, fillcolor=(255,255,255), resample=Image.BICUBIC)
    return img

def is_blank(image, threshold=BLANK_THRESHOLD) -> bool:
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold

def pdf_to_pages(path: Path, zoom=PDF_ZOOM) -> list:
    doc = fitz.open(path); matrix = fitz.Matrix(zoom, zoom); pages = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        pages.append({'index': i, 'image': resize(deskew(Image.frombytes('RGB', [pix.width, pix.height], pix.samples)))})
    doc.close()
    return pages

def parse_json(text: str) -> dict:
    try:
        m = re.search(r'\{.*\}', text, re.S)
        return json.loads(m.group()) if m else {}
    except Exception:
        return {}

def apply_template(messages) -> str:
    try:
        return processor.apply_chat_template(messages, tokenize=False,
            add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _decode(out_i, in_len):
    return processor.decode(out_i[in_len:], skip_special_tokens=True,
                            clean_up_tokenization_spaces=False)

def ask_single(prompt, image) -> dict:
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    inputs = processor(text=[apply_template(msgs)], images=[image], return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    return {'text': _decode(out[0], inputs['input_ids'].shape[1]),
            'tokens_in': int(inputs['input_ids'].shape[1]),
            'tokens_out': int(out[0].shape[0] - inputs['input_ids'].shape[1]),
            'elapsed': round(time.time()-t0, 2)}

def ask_batch(prompt, images) -> list:
    if not images: return []
    if len(images) == 1: return [ask_single(prompt, images[0])]
    msgs = [[{'role':'user','content':[{'type':'image','image':img},{'type':'text','text':prompt}]}] for img in images]
    inputs = processor(text=[apply_template(m) for m in msgs], images=images,
                       return_tensors='pt', padding=True).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    el = time.time() - t0
    in_len = inputs['input_ids'].shape[1]
    attn = inputs.get('attention_mask')
    return [{'text': _decode(out[i], in_len),
             'tokens_in': int(attn[i].sum().item()) if attn is not None else in_len,
             'tokens_out': int(out[i].shape[0] - in_len), 'elapsed': round(el/len(images), 2)}
            for i in range(len(images))]

print('✅ Utilitaires OK')

## Cellule 6 — Schémas + Prompt (TCR avec DEBIT/CREDIT séparés)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 6 — SCHÉMAS + PROMPT | BILANS_V8 | 2026-08-15
# TCR : 4 colonnes debit_n/credit_n/debit_n1/credit_n1
# ════════════════════════════════════════════════════════════
SCHEMAS = {
 'ACTIF': {'cols': ['brut','amort','n','n1'], 'postes': {
   'ecarts_acquisition_goodwill': 'Ecart d acquisition goodwill',
   'immobilisations_incorporelles': 'Immobilisations incorporelles',
   'terrains': 'Terrains',
   'batiments': 'Batiments',
   'autres_immobilisations_corporelles': 'Autres Immobilisations corporelles',
   'immobilisations_en_concession': 'Immobilisations en concession',
   'immobilisations_en_cours': 'Immobilisations en cours',
   'titres_mis_en_equivalence': 'Titres mis en equivalence',
   'autres_participations_creances': 'Autres participations et creances rattachees',
   'autres_titres_immobilises': 'Autres titres immobilises',
   'prets_actifs_financiers_non_courants': 'Prets et autres actifs financiers non courants',
   'impots_differes_actif': 'Impots Differes Actif',
   'total_actif_non_courant': 'TOTAL ACTIF NON COURANT',
   'stocks_encours': 'Stocks et encours',
   'clients': 'Clients',
   'autres_debiteurs': 'Autres debiteurs',
   'impots_assimiles_actif': 'Impots & Assimiles',
   'autres_creances_assimiles': 'Autres Creances & Emplois assimiles',
   'placements_financiers_courants': 'Placements et autres actifs financiers courants',
   'tresorerie_actif': 'Tresorerie',
   'total_actif_courant': 'TOTAL ACTIF COURANT',
   'total_general_actif': 'TOTAL GENERAL ACTIF'}},
 'PASSIF': {'cols': ['n','n1'], 'postes': {
   'capital_emis': 'Capital emis (ou compte de l exploitant)',
   'capital_non_appele': 'Capital non appele',
   'primes_reserves': 'Primes et reserves (Reserves consolidees)',
   'ecart_reevaluation': 'Ecart de reevaluation',
   'ecart_equivalence': 'Ecart d equivalence (1)',
   'resultat_net_passif': 'Resultat net (Resultat net part du groupe) (1)',
   'report_a_nouveau': 'Autres capitaux propres - Report a nouveau',
   'part_societe_consolidante': 'Part de la societe consolidante (1)',
   'part_minoritaires': 'Part des minoritaires (1)',
   'total_capitaux_propres': 'TOTAL I',
   'emprunts_dettes_financieres': 'Emprunts et dettes financieres',
   'impots_differes_provisionnes': 'Impots differes et provisionnes',
   'autres_dettes_non_courantes': 'Autres dettes non courantes',
   'provisions_produits_avance': 'Provisions et produits comptabilises d avance',
   'total_passifs_non_courants': 'TOTAL PASSIFS NON COURANTS II',
   'fournisseurs_rattaches': 'Fournisseurs et comptes rattaches',
   'impots_passif': 'Impots',
   'autres_dettes': 'Autres dettes',
   'tresorerie_passif': 'Tresorerie Passif',
   'total_passifs_courants': 'TOTAL PASSIFS COURANTS (II ou III)',
   'total_general_passif': 'TOTAL GENERAL PASSIF'}},
 'TCR': {'cols': ['debit_n','credit_n','debit_n1','credit_n1'], 'postes': {
   'ventes_marchandises': 'Ventes de Marchandises',
   'produits_fabriques': 'Produits Fabriques',
   'prestations_services': 'Prestations de Services',
   'ventes_travaux': 'Ventes de Travaux',
   'produits_annexes': 'Produits Annexes',
   'rabais_remises_ristournes_accordes': 'Rabais, remises, ristournes accordes',
   'chiffre_affaires_net': 'Chiffre d affaires net des Rabais, remises, ristournes',
   'production_stockee_destockee': 'Production Stockee ou destockee',
   'production_immobilisee': 'Production immobilisee',
   'subvention_exploitation': 'Subvention d exploitation',
   'production_exercice': 'I-Production de l exercice',
   'achats_marchandises_vendues': 'Achats de Marchandises vendues',
   'matieres_premieres': 'Matieres premieres',
   'autres_approvisionnements': 'Autres Approvisionnements',
   'variation_stocks': 'Variation des Stocks',
   'achats_etudes_prestations': 'Achats d Etudes et de Prestations de services',
   'autres_consommations': 'Autres consommations',
   'rabais_remises_obtenus_achats': 'Rabais, remises, ristournes obtenus sur achats',
   'sous_traitance_generale': 'Sous-traitance generale',
   'locations': 'Locations',
   'entretien_reparations': 'Entretien, reparations et maintenance',
   'primes_assurances': 'Primes d assurances',
   'personnel_exterieur': 'Personnel exterieur a l entreprise',
   'remuneration_intermediaires': 'Remuneration d intermediaires et honoraires',
   'publicite': 'Publicite',
   'deplacements_missions': 'Deplacements, missions et receptions',
   'autres_services': 'Autres services',
   'rabais_remises_obtenus_services': 'Rabais, remises, ristournes obtenus sur services exterieurs',
   'consommations_exercice': 'II-Consommations de l exercice',
   'valeur_ajoutee_exploitation': 'III-Valeur ajoutee d exploitation (I-II)',
   'charges_personnel': 'Charges de personnel',
   'impots_taxes_assimiles': 'Impots et taxes et versements assimiles',
   'excedent_brut_exploitation': 'IV-Excedent brut d exploitation',
   'autres_produits_operationnels': 'Autres produits operationnels',
   'autres_charges_operationnelles': 'Autres charges operationnelles',
   'dotations_amortissements': 'Dotations aux amortissements',
   'provisions': 'Provisions',
   'pertes_valeur': 'Perte de Valeur',
   'reprises_pertes_valeur_provisions': 'Reprise sur pertes de valeur et provisions',
   'resultat_operationnel': 'V-Resultat operationnel',
   'produits_financiers': 'Produits financiers',
   'charges_financieres': 'Charges financieres',
   'resultat_financier': 'VI-Resultat Financier',
   'resultat_ordinaire': 'VII-Resultat ordinaire (V+VI)',
   'elements_extraordinaires_produits': 'Elements extraordinaires (Produits)',
   'elements_extraordinaires_charges': 'Elements extraordinaires (Charges)',
   'resultat_extraordinaire': 'VIII-Resultat extraordinaire',
   'impots_exigibles_resultats': 'Impots exigibles sur resultats',
   'impots_differes_resultats': 'Impots differes (variations) sur resultats',
   'resultat_net_exercice': 'RESULTAT NET DE L EXERCICE'}},
}

# ── Prompt généré ──
L = []
L.append('Lis cette page d\'un dossier fiscal algerien (imprime Serie G).')
L.append('')
L.append('ETAPE 1 — Identifie le tableau principal :')
L.append('- ACTIF  : titre "BILAN (ACTIF)"')
L.append('- PASSIF : titre "BILAN (PASSIF)"')
L.append('- TCR    : titre "COMPTE DE RESULTAT"')
L.append('- DECL   : page "DECLARATION DE L\'IMPOT SUR LES BENEFICES DES SOCIETES"')
L.append('- AUTRE  : toute autre page')
L.append('')
L.append('ETAPE 2 — Extrais les montants en JSON :')
L.append('{"type": ..., "entreprise": ..., "exercice": ..., "nif": ..., "postes": {cle: {col: montant}}}')
L.append('- ACTIF : colonnes brut, amort, n, n1')
L.append('- PASSIF : colonnes n, n1')
L.append('- TCR : 4 colonnes : debit_n, credit_n, debit_n1, credit_n1')
L.append('  * Lis EXACTEMENT dans quelle colonne (DEBIT ou CREDIT) se trouve chaque montant')
L.append('  * Une valeur ne peut etre que dans UNE seule colonne. Si case vide : null')
L.append('- Montant entre parentheses = negatif : (1 553 799) → -1553799')
L.append('- Montants en NOMBRES JSON sans espaces ; case vide : null')
L.append('- Utilise EXACTEMENT les cles ci-dessous. Si AUTRE : {"type": "AUTRE"}')
for table, spec in SCHEMAS.items():
    L.append(f'--- Si {table} ---')
    for key, label in spec['postes'].items():
        L.append(f'{key} : ligne "{label}"')
L.append('')
L.append('REGLES : JSON valide uniquement, sans texte avant/apres, aucun champ invente.')
PROMPT_BILAN = '\n'.join(L)

PROMPT_CLASSIF = ('Page dun dossier fiscal algerien Serie G. Reponds UN seul mot : '
                  'ACTIF si titre BILAN (ACTIF) ; PASSIF si BILAN (PASSIF) ; '
                  'TCR si COMPTE DE RESULTAT ; DECL si page DECLARATION ; AUTRE sinon.')

print(f'✅ Schémas + prompt OK ({len(PROMPT_BILAN)} caractères)')

## Cellule 7 — Normalisation

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 7 — NORMALISATION | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
def norm_montant(v):
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    neg = (s.startswith('(') and s.endswith(')')) or s.startswith('-')
    s = re.sub(r'[^\d.,]', '', s)
    if not s: return None
    if s.count(',') == 1 and '.' not in s: s = s.replace(',', '.')
    elif ',' in s: s = s.replace(',', '')
    elif s.count('.') > 1: s = s.replace('.', '')
    try: return -float(s) if neg else float(s)
    except Exception: return None

def norm_str(v):
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).strip())
    return s if s and s.lower() not in ('null','none','n/a') else None

def norm_upper(v):
    s = norm_str(v)
    return s.upper() if s else None

def norm_nif(v):
    s = norm_str(v)
    return re.sub(r'[^0-9]', '', s) if s else None

def norm_annee4(v):
    s = norm_str(v) or ''
    m = re.findall(r'20\d{2}', s)
    return m[-1] if m else None

def normalise_table(table, data):
    spec = SCHEMAS[table]
    raw = data.get('postes') or {}
    out = {}
    for key in spec['postes']:
        vals = raw.get(key)
        vals = vals if isinstance(vals, dict) else {}
        for col in spec['cols']:
            out[f'{key}_{col}'] = norm_montant(vals.get(col))
    return out

print('✅ Normalisation OK')

## Cellule 8 — Cohérence inter-pages

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 8 — COHÉRENCE INTER-PAGES | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
def norm_identite(s):
    s = norm_upper(s)
    return re.sub(r'[^A-Z0-9]', '', s) if s else None

def meme_entreprise(a, b):
    if not a or not b: return True
    if a == b or a in b or b in a: return True
    return difflib.SequenceMatcher(None, a, b).ratio() > 0.80

def controle_coherence(parsed, annee_attendue=None):
    anomalies, excluded = [], set()
    idents = [(p['index'], norm_identite(p['meta'].get('entreprise'))) for p in parsed]
    vals = [e for _, e in idents if e]
    ref_ent = max(set(vals), key=lambda e: sum(1 for x in vals if meme_entreprise(e, x))) if vals else None
    for idx, e in idents:
        if ref_ent and e and not meme_entreprise(ref_ent, e):
            anomalies.append(f'PAGE {idx+1}: CLIENT DIFFERENT')
            excluded.add(idx)
    annees = [(p['index'], norm_annee4(p['meta'].get('exercice'))) for p in parsed]
    yy = [a for _, a in annees if a]
    ref_annee = max(set(yy), key=yy.count) if yy else None
    for idx, a in annees:
        if ref_annee and a and a != ref_annee:
            anomalies.append(f'PAGE {idx+1}: ANNEE {a} ≠ {ref_annee}')
    if annee_attendue and ref_annee and ref_annee != str(annee_attendue):
        anomalies.append(f'ANNEE DOSSIER {ref_annee} ≠ ATTENDUE {annee_attendue}')
    nifs = [(p['index'], norm_nif(p['meta'].get('nif'))) for p in parsed]
    nn = [n for _, n in nifs if n and len(n) >= 12]
    ref_nif = max(set(nn), key=nn.count) if nn else None
    for idx, n in nifs:
        if ref_nif and n and len(n) >= 12 and n != ref_nif:
            anomalies.append(f'PAGE {idx+1}: NIF DIFFERENT')
    return ref_ent, ref_nif, ref_annee, anomalies, excluded

print('✅ Contrôle cohérence OK')

## Cellule 9 — Export Excel récapitulatif

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 9 — EXPORT EXCEL | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
COLORS = {'META':'FFD6E4F0', 'ACTIF':'FFE2EFDA', 'PASSIF':'FFDAE3F3', 'TCR':'FFFCE4D6'}
HDRS   = {'META':'FF1F4E79', 'ACTIF':'FF375623', 'PASSIF':'FF203864', 'TCR':'FF833C00'}

def build_cols():
    cols = [('META', k, lab) for k, lab in [
        ('fichier','Fichier'), ('entreprise','Entreprise'), ('nif','NIF'),
        ('exercice','Exercice'), ('annee_exercice','Année exercice'),
        ('annee_depot','Année dépôt'), ('date_traitement','Date traitement'),
        ('temps_total_s','Temps (s)'), ('pages_trouvees','Tableaux trouvés'),
        ('anomalies','Anomalies')]]
    for key in SCHEMAS['ACTIF']['postes']: cols.append(('ACTIF', key, key))
    for key in SCHEMAS['PASSIF']['postes']: cols.append(('PASSIF', key, key))
    for key in SCHEMAS['TCR']['postes']: cols.append(('TCR', key, key))
    return cols

def val_for(d, groupe, key, suf):
    t = d.get(groupe) or {}
    if groupe == 'TCR':
        return t.get(f'{key}_debit_{suf}'), t.get(f'{key}_credit_{suf}')
    if groupe == 'ACTIF' and (key.endswith('_brut') or key.endswith('_amort')):
        return t.get(key) if suf == 'n' else None
    return t.get(f'{key}_{suf}')

def create_excel(path, rows):
    all_cols = build_cols()
    wb = Workbook(); ws = wb.active; ws.title = 'Bilans'
    grp = defaultdict(list); idx = 1
    for g, _, _ in all_cols: grp[g].append(idx); idx += 1
    for g, cs in grp.items():
        s, e = cs[0], cs[-1]
        if s < e: ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)
        c = ws.cell(row=1, column=s); c.value = g
        c.font = Font(bold=True, color='FFFFFFFF', name='Arial', size=11)
        c.fill = PatternFill('solid', start_color=HDRS[g])
    for i, (g, _, label) in enumerate(all_cols, start=1):
        c = ws.cell(row=2, column=i); c.value = label
        c.font = Font(bold=True, name='Arial', size=8)
        c.fill = PatternFill('solid', start_color=COLORS[g])
        ws.column_dimensions[get_column_letter(i)].width = 16
    ws.freeze_panes = ws.cell(row=3, column=7)
    rn = 3
    for d in rows:
        annee_n = norm_str(d.get('exercice'))
        annee_n1 = str(int(annee_n)-1) if annee_n and annee_n.isdigit() else None
        for exer, annee, suf in [('N', annee_n, 'n'), ('N-1', annee_n1, 'n1')]:
            for ci, (g, key, _) in enumerate(all_cols, start=1):
                if g == 'META':
                    val = exer if key == 'exercice' else (annee if key == 'annee_exercice' else d.get(key))
                else:
                    val = val_for(d, g, key, suf)
                c = ws.cell(row=rn, column=ci); c.value = val
                c.font = Font(name='Arial', size=9)
                c.fill = PatternFill('solid', start_color=COLORS[g])
                if isinstance(val, float): c.number_format = '#,##0.00'
            rn += 1
    wb.save(path)
    print(f'✅ Excel : {path} | {len(rows)} bilans')

print('✅ Export Excel OK')

## Cellule 10 — Log

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 10 — LOG | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
def log(msg: str):
    ligne = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} — {msg}"
    print(ligne)
    with open(LOG_PATH, 'a', encoding='utf-8') as f: f.write(ligne + '\n')
print('✅ Log OK')

## Cellule 11 — Pipeline (1 PDF = 1 bilan)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 11 — PIPELINE | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
deja = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter = [p for p in pdfs if p.stem not in deja]
log(f'À traiter : {len(a_traiter)} | déjà traités : {len(deja)}')

t_total = time.time(); n_ok = n_err = 0
TYPES = {'ACTIF', 'PASSIF', 'TCR'}

def parse_type(text):
    t = (text or '').upper()
    for k in ('ACTIF', 'PASSIF', 'TCR', 'DECL'):
        if k in t: return k
    return 'AUTRE'

for num, pdf_path in enumerate(a_traiter, start=1):
    t0 = time.time()
    try:
        pages   = pdf_to_pages(pdf_path)
        actives = [p for p in pages if not is_blank(p['image'])]
        tok_in = tok_out = 0

        mini = [resize(p['image'], 600) for p in actives]
        reps1 = []
        for bs in range(0, len(mini), 16):
            reps1 += ask_batch(PROMPT_CLASSIF, mini[bs:bs+16])
        tok_in  += sum(r['tokens_in']  for r in reps1)
        tok_out += sum(r['tokens_out'] for r in reps1)
        utiles = [(p, parse_type(r['text'])) for p, r in zip(actives, reps1)
                  if parse_type(r['text']) != 'AUTRE']

        parsed, annee_depot = [], None
        for bs in range(0, len(utiles), GPU_BATCH_SIZE):
            batch = utiles[bs:bs+GPU_BATCH_SIZE]
            reps  = ask_batch(PROMPT_BILAN, [p['image'] for p, _ in batch])
            for (page, tpage), rep in zip(batch, reps):
                tok_in += rep['tokens_in']; tok_out += rep['tokens_out']
                data = parse_json(rep['text'])
                t = data.get('type', tpage)
                if t not in TYPES and t != 'DECL': continue
                meta = {k: data.get(k) for k in ('entreprise', 'nif')}
                meta['exercice'] = None if t == 'DECL' else data.get('exercice')
                if t == 'DECL':
                    dep = norm_annee4(data.get('annee_souscription'))
                    if dep and not annee_depot: annee_depot = dep
                parsed.append({'index': page['index'], 'type': t, 'meta': meta, 'data': data})
            gc.collect(); torch.cuda.empty_cache()

        ref_ent, ref_nif, ref_annee, anomalies, excluded = controle_coherence(parsed, ANNEE_ATTENDUE)
        ent_raw = next((norm_str(p['meta'].get('entreprise')) for p in parsed
                        if p['index'] not in excluded
                        and meme_entreprise(ref_ent or '', norm_identite(p['meta'].get('entreprise')))), None)
        tables, tcr_pages, doublons = {}, [], []
        for p in parsed:
            if p['index'] in excluded or p['type'] == 'DECL': continue
            t = p['type']
            if t == 'TCR': tcr_pages.append(p['data'])
            else:
                if t in tables: doublons.append(t); continue
                tables[t] = normalise_table(t, p['data'])
        tcr = {}
        for d in tcr_pages:
            for k, v in normalise_table('TCR', d).items():
                if tcr.get(k) is None and v is not None: tcr[k] = v
        if tcr: tables['TCR'] = tcr
        if doublons: anomalies.append('DOUBLON: ' + ', '.join(doublons))

        dt = round(time.time() - t0, 2)
        result = {
            'fichier': pdf_path.name,
            'entreprise': ent_raw, 'nif': ref_nif,
            'exercice': ref_annee, 'annee_depot': annee_depot,
            'date_traitement': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'temps_total_s': dt, 'tokens_in': tok_in, 'tokens_out': tok_out,
            'pages_trouvees': ', '.join(sorted(tables.keys())),
            'anomalies': ' | '.join(anomalies) if anomalies else None,
            **{t: tables.get(t, {}) for t in TYPES},
        }
        with open(JSON_DIR / f'{pdf_path.stem}.json', 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2, default=str)
        n_ok += 1
        eta = (time.time() - t_total) / num * (len(a_traiter) - num)
        msg = f'[{num: >4}/{len(a_traiter)}] ✅ {pdf_path.name} | {dt:.1f}s | {result["pages_trouvees"]}'
        if result['anomalies']: msg += f' | 🔴 {result["anomalies"]}'
        log(msg + f' | ETA {eta/3600:.1f}h')
    except Exception as e:
        n_err += 1
        log(f'[{num: >4}/{len(a_traiter)}] ❌ {pdf_path.name} — {e}')
        continue

log('Génération Excel...')
rows = [json.load(open(jf, encoding='utf-8')) for jf in sorted(JSON_DIR.glob('*.json'))]
create_excel(EXCEL_PATH, rows)
log(f'✅ Terminé en {time.time()-t_total:.1f}s | OK {n_ok} | Erreurs {n_err}')

## Cellule 12 — Contrôles comptables

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 12 — CONTRÔLES COMPTABLES | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
for d in rows:
    a, p, t = d.get('ACTIF') or {}, d.get('PASSIF') or {}, d.get('TCR') or {}
    for suf, lab in [('n', 'N'), ('n1', 'N-1')]:
        ta, tp = a.get(f'total_general_actif_{suf}'), p.get(f'total_general_passif_{suf}')
        if ta and tp and abs(ta - tp) > 1:
            print(f'❌ {d["fichier"]} [{lab}] : Actif {ta:,.0f} ≠ Passif {tp:,.0f}')
    rn_credit = t.get('resultat_net_exercice_credit_n')
    rn_debit  = t.get('resultat_net_exercice_debit_n')
    rn_passif = p.get('resultat_net_passif_n')
    rn_tcr = (rn_credit or 0) - (rn_debit or 0)
    if rn_passif is not None and (rn_credit is not None or rn_debit is not None):
        if abs(rn_tcr - rn_passif) > 1:
            print(f'❌ {d["fichier"]} : RN TCR {rn_tcr:,.0f} ≠ Passif {rn_passif:,.0f}')
print('🔎 Contrôles terminés')

## Cellule 13 — Générateur FORFAIT avec contrôle de cohérence (ROUGE si incohérence)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 13 — GÉNÉRATEUR FORFAIT | BILANS_V8 | 2026-08-15
# Contrôle : si valeur_extraite ≠ formule → ROUGE + commentaire
# ════════════════════════════════════════════════════════════
def to_kdzd(v):
    if v is None: return None
    return round(v / 1000.0, 2)

def _clean_name(s):
    return re.sub(r'[^A-Za-z0-9 _-]', '', s or 'ENTREPRISE').strip()

def _safe_sum(*values):
    vals = [v for v in values if v is not None]
    return sum(vals) if vals else None

def _write_cell(ws, row, col, value, incoherent=False, detail=''):
    if value is None: return
    cell = ws.cell(row=row, column=col, value=value)
    if incoherent:
        cell.font = RED_FONT
        cell.fill = RED_FILL
        cell.border = RED_BORDER
        if detail:
            cell.comment = Comment(detail, 'BILANS_V8')
    else:
        cell.font = NORMAL_FONT

def generate_forfait(result):
    if not TEMPLATE_FORFAIT.exists():
        print(f'❌ Template introuvable : {TEMPLATE_FORFAIT}')
        return None

    entreprise = _clean_name(result.get('entreprise'))
    annee = result.get('exercice') or 'XXXX'
    out_path = FORFAIT_OUTPUT_DIR / f'FORFAIT Cas 1_{entreprise}_{annee}.xlsx'

    try:
        shutil.copy2(TEMPLATE_FORFAIT, out_path)
        wb = load_workbook(out_path)
    except Exception as e:
        print(f'❌ Erreur copie : {e}')
        return None

    actif  = result.get('ACTIF')  or {}
    passif = result.get('PASSIF') or {}
    tcr    = result.get('TCR')    or {}
    n_filled = 0
    n_incoherent = 0

    # ── SAISIE ACTIF ──
    cfg = FORFAIT_COORDS['actif']
    if cfg['sheet'] in wb.sheetnames:
        ws = wb[cfg['sheet']]
        r, c = cfg['nature_bilan']
        ws.cell(row=r, column=c, value='Fiscal')
        exerc = result.get('exercice')
        if exerc:
            r, c = cfg['date_arrete_n']
            ws.cell(row=r, column=c, value=exerc)

        cn, cn1 = cfg['cols_n'], cfg['cols_n1']
        for key, row in cfg['rows'].items():
            brut  = actif.get(f'{key}_brut')
            amort = actif.get(f'{key}_amort')
            net   = actif.get(f'{key}_n')
            net_n1= actif.get(f'{key}_n1')

            # Contrôle Net = Brut - Amort
            incoh = False
            detail = ''
            if brut is not None and net is not None:
                calc = brut - (amort or 0)
                if abs(calc - net) > TOLERANCE_KDZD * 1000:
                    incoh = True
                    detail = (f'INCOHÉRENCE: Net ≠ Brut - Amort\n'
                              f'Net extrait: {net:,.2f}\n'
                              f'Calculé: {calc:,.2f}\n'
                              f'Écart: {abs(calc-net):,.2f}')
                    n_incoherent += 1

            _write_cell(ws, row, cn['brut'],  to_kdzd(brut), incoh, detail)
            _write_cell(ws, row, cn['amort'], to_kdzd(amort), incoh, detail)
            _write_cell(ws, row, cn['net'],   to_kdzd(net), incoh, detail)
            _write_cell(ws, row, cn1['net'],  to_kdzd(net_n1))
            n_filled += 1

    # ── SAISIE PASSIF ──
    cfg = FORFAIT_COORDS['passif']
    if cfg['sheet'] in wb.sheetnames:
        ws = wb[cfg['sheet']]
        for key, row in cfg['rows'].items():
            net   = passif.get(f'{key}_n')
            net_n1= passif.get(f'{key}_n1')
            _write_cell(ws, row, cfg['col_n'],  to_kdzd(net))
            _write_cell(ws, row, cfg['col_n1'], to_kdzd(net_n1))
            n_filled += 1

    # ── SAISIE TCR ──
    cfg = FORFAIT_COORDS['tcr']
    if cfg['sheet'] in wb.sheetnames:
        ws = wb[cfg['sheet']]
        ctn, ctn1 = cfg['cols_n'], cfg['cols_n1']
        for key, row in cfg['rows'].items():
            deb_n  = tcr.get(f'{key}_debit_n')
            cred_n = tcr.get(f'{key}_credit_n')
            deb_n1 = tcr.get(f'{key}_debit_n1')
            cred_n1= tcr.get(f'{key}_credit_n1')
            _write_cell(ws, row, ctn['debit'],  to_kdzd(deb_n))
            _write_cell(ws, row, ctn['credit'], to_kdzd(cred_n))
            _write_cell(ws, row, ctn1['debit'],  to_kdzd(deb_n1))
            _write_cell(ws, row, ctn1['credit'], to_kdzd(cred_n1))
            n_filled += 1

    wb.save(out_path)
    statut = '✅' if n_incoherent == 0 else f'⚠️ {n_incoherent} incohérence(s) en ROUGE'
    print(f'{statut} | {out_path.name} | {n_filled} cellules')
    return out_path

print('✅ Générateur FORFAIT OK')

## Cellule 14 — Exécution génération FORFAIT

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 14 — EXÉCUTION FORFAIT | BILANS_V8 | 2026-08-15
# ════════════════════════════════════════════════════════════
json_files = sorted(JSON_DIR.glob('*.json'))
if not json_files:
    print('⚠️ Aucun JSON — exécute d\'abord la Cellule 11')
else:
    print(f'📁 {len(json_files)} JSON → FORFAIT')
    n_gen = n_err = 0
    for jf in json_files:
        try:
            data = json.load(open(jf, encoding='utf-8'))
            if not (data.get('ACTIF') or data.get('PASSIF') or data.get('TCR')):
                print(f'⏭️  {jf.stem} : vide → ignoré')
                continue
            if generate_forfait(data):
                n_gen += 1
        except Exception as e:
            print(f'❌ {jf.stem} : {e}')
            n_err += 1
    print(f'\n✅ {n_gen} FORFAIT générés | {n_err} erreurs → {FORFAIT_OUTPUT_DIR}')

## Cellule 15 — Vérification coordonnées (à exécuter UNE FOIS)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 15 — VÉRIFICATION COORDONNÉES | BILANS_V8 | 2026-08-15
# À exécuter UNE FOIS pour vérifier FORFAIT_COORDS
# ════════════════════════════════════════════════════════════
if not TEMPLATE_FORFAIT.exists():
    print(f'⚠️ Template introuvable : {TEMPLATE_FORFAIT}')
else:
    wb = load_workbook(TEMPLATE_FORFAIT, data_only=False)
    print(f'Feuilles : {wb.sheetnames}\n')
    for sheet_name in ['Saisie actif', 'Saisie passif', 'Saisie TCR']:
        if sheet_name not in wb.sheetnames:
            print(f'⚠️ Feuille absente : {sheet_name}')
            continue
        ws = wb[sheet_name]
        print(f'=== {sheet_name} ({ws.max_row} lignes × {ws.max_column} cols) ===')
        for r in range(1, min(ws.max_row, 60) + 1):
            row_vals = []
            for c in range(1, min(ws.max_column, 10) + 1):
                v = ws.cell(row=r, column=c).value
                if v is not None and str(v).strip() != '':
                    row_vals.append((c, str(v)[:35]))
            if row_vals:
                print(f'  Ligne {r:3d}: {row_vals}')
        print()
    wb.close()
    print('✅ Vérification terminée — ajustez FORFAIT_COORDS si nécessaire')